In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, FloatType, IntegerType, TimestampType

In [0]:
spark = (SparkSession.builder
    .appName("Day1-Exercise")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate())

In [0]:
raw = [
    ("FC-01", "2024-01-01 08:00", 720.5, 1.02, 0.78),
    ("FC-01", "2024-01-01 09:00", 735.2, 1.05, 0.75),
    ("FC-01", "2024-01-01 10:00", 698.1, 0.98, 0.81),
    ("FC-02", "2024-01-01 08:00", 715.0, 1.01, 0.79),
    ("FC-02", "2024-01-01 09:00", 729.8, 1.04, 0.76),
    ("FC-02", "2024-01-01 10:00", 701.3, 0.99, 0.80),
    ("FC-03", "2024-01-01 08:00", 740.2, 1.06, 0.74),
    ("FC-03", "2024-01-01 09:00", 755.1, 1.08, 0.71),
    ("FC-03", "2024-01-01 10:00", 688.4, 0.97, 0.83),
    ("FC-04", "2024-01-01 08:00", 710.6, 1.00, 0.80),
    ("FC-04", "2024-01-01 09:00", 722.3, 1.03, 0.77),
    ("FC-04", "2024-01-01 10:00", 695.8, 0.97, 0.82),
    ("FC-05", "2024-01-01 08:00", 750.9, 1.07, 0.72),
    ("FC-05", "2024-01-01 09:00", 762.4, 1.09, 0.70),
    ("FC-05", "2024-01-01 10:00", 681.2, 0.96, 0.85),
]

In [0]:
schema = StructType([
    StructField("cell_id",    StringType(), nullable=False),
    StructField("timestamp",  StringType(), nullable=True),
    StructField("temp_c",     FloatType(),  nullable=True),
    StructField("pressure",   FloatType(),  nullable=True),
    StructField("voltage",    FloatType(),  nullable=True),
])

In [0]:
df = spark.createDataFrame(raw, schema=schema)
df.printSchema()
df.show()

root
 |-- cell_id: string (nullable = false)
 |-- timestamp: string (nullable = true)
 |-- temp_c: float (nullable = true)
 |-- pressure: float (nullable = true)
 |-- voltage: float (nullable = true)

+-------+----------------+------+--------+-------+
|cell_id|       timestamp|temp_c|pressure|voltage|
+-------+----------------+------+--------+-------+
|  FC-01|2024-01-01 08:00| 720.5|    1.02|   0.78|
|  FC-01|2024-01-01 09:00| 735.2|    1.05|   0.75|
|  FC-01|2024-01-01 10:00| 698.1|    0.98|   0.81|
|  FC-02|2024-01-01 08:00| 715.0|    1.01|   0.79|
|  FC-02|2024-01-01 09:00| 729.8|    1.04|   0.76|
|  FC-02|2024-01-01 10:00| 701.3|    0.99|    0.8|
|  FC-03|2024-01-01 08:00| 740.2|    1.06|   0.74|
|  FC-03|2024-01-01 09:00| 755.1|    1.08|   0.71|
|  FC-03|2024-01-01 10:00| 688.4|    0.97|   0.83|
|  FC-04|2024-01-01 08:00| 710.6|     1.0|    0.8|
|  FC-04|2024-01-01 09:00| 722.3|    1.03|   0.77|
|  FC-04|2024-01-01 10:00| 695.8|    0.97|   0.82|
|  FC-05|2024-01-01 08:00| 750.9| 

In [0]:
df.show()

+-------+----------------+------+--------+-------+
|cell_id|       timestamp|temp_c|pressure|voltage|
+-------+----------------+------+--------+-------+
|  FC-01|2024-01-01 08:00| 720.5|    1.02|   0.78|
|  FC-01|2024-01-01 09:00| 735.2|    1.05|   0.75|
|  FC-01|2024-01-01 10:00| 698.1|    0.98|   0.81|
|  FC-02|2024-01-01 08:00| 715.0|    1.01|   0.79|
|  FC-02|2024-01-01 09:00| 729.8|    1.04|   0.76|
|  FC-02|2024-01-01 10:00| 701.3|    0.99|    0.8|
|  FC-03|2024-01-01 08:00| 740.2|    1.06|   0.74|
|  FC-03|2024-01-01 09:00| 755.1|    1.08|   0.71|
|  FC-03|2024-01-01 10:00| 688.4|    0.97|   0.83|
|  FC-04|2024-01-01 08:00| 710.6|     1.0|    0.8|
|  FC-04|2024-01-01 09:00| 722.3|    1.03|   0.77|
|  FC-04|2024-01-01 10:00| 695.8|    0.97|   0.82|
|  FC-05|2024-01-01 08:00| 750.9|    1.07|   0.72|
|  FC-05|2024-01-01 09:00| 762.4|    1.09|    0.7|
|  FC-05|2024-01-01 10:00| 681.2|    0.96|   0.85|
+-------+----------------+------+--------+-------+



In [0]:
df_hot = df.filter('temp_c > 730.0').select("cell_id",'temp_c')
df_hot.show()

+-------+------+
|cell_id|temp_c|
+-------+------+
|  FC-01| 735.2|
|  FC-03| 740.2|
|  FC-03| 755.1|
|  FC-05| 750.9|
|  FC-05| 762.4|
+-------+------+



In [0]:
df_with_f = df.withColumn(colName = 'temp_f', col = F.round(((df['temp_c'] * (9/5)) +32),1))
df_with_f.select("cell_id",'temp_c','temp_f').show()

+-------+------+------+
|cell_id|temp_c|temp_f|
+-------+------+------+
|  FC-01| 720.5|1328.9|
|  FC-01| 735.2|1355.4|
|  FC-01| 698.1|1288.6|
|  FC-02| 715.0|1319.0|
|  FC-02| 729.8|1345.6|
|  FC-02| 701.3|1294.3|
|  FC-03| 740.2|1364.4|
|  FC-03| 755.1|1391.2|
|  FC-03| 688.4|1271.1|
|  FC-04| 710.6|1311.1|
|  FC-04| 722.3|1332.1|
|  FC-04| 695.8|1284.4|
|  FC-05| 750.9|1383.6|
|  FC-05| 762.4|1404.3|
|  FC-05| 681.2|1258.2|
+-------+------+------+



In [0]:
df.groupBy("cell_id").count().orderBy("cell_id").show()

+-------+-----+
|cell_id|count|
+-------+-----+
|  FC-01|    3|
|  FC-02|    3|
|  FC-03|    3|
|  FC-04|    3|
|  FC-05|    3|
+-------+-----+



In [0]:
df_with_f.write.saveAsTable('test.test_schema.fc_readings_data')

In [0]:
df_back = spark.read.table("test.test_schema.fc_readings_data")
df_back.show()

+-------+----------------+------+--------+-------+------+
|cell_id|       timestamp|temp_c|pressure|voltage|temp_f|
+-------+----------------+------+--------+-------+------+
|  FC-01|2024-01-01 08:00| 720.5|    1.02|   0.78|1328.9|
|  FC-01|2024-01-01 09:00| 735.2|    1.05|   0.75|1355.4|
|  FC-01|2024-01-01 10:00| 698.1|    0.98|   0.81|1288.6|
|  FC-02|2024-01-01 08:00| 715.0|    1.01|   0.79|1319.0|
|  FC-02|2024-01-01 09:00| 729.8|    1.04|   0.76|1345.6|
|  FC-02|2024-01-01 10:00| 701.3|    0.99|    0.8|1294.3|
|  FC-03|2024-01-01 08:00| 740.2|    1.06|   0.74|1364.4|
|  FC-03|2024-01-01 09:00| 755.1|    1.08|   0.71|1391.2|
|  FC-03|2024-01-01 10:00| 688.4|    0.97|   0.83|1271.1|
|  FC-04|2024-01-01 08:00| 710.6|     1.0|    0.8|1311.1|
|  FC-04|2024-01-01 09:00| 722.3|    1.03|   0.77|1332.1|
|  FC-04|2024-01-01 10:00| 695.8|    0.97|   0.82|1284.4|
|  FC-05|2024-01-01 08:00| 750.9|    1.07|   0.72|1383.6|
|  FC-05|2024-01-01 09:00| 762.4|    1.09|    0.7|1404.3|
|  FC-05|2024-